<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

# ¿Por qué utilizar SVM para este problema?

El modelo Support Vector Machine (SVM) es un algoritmo de clasificación supervisada que busca encontrar la frontera óptima que mejor separa las distintas clases del problema.

En este proyecto, el objetivo es clasificar el riesgo Urban Heat Island (UHI) (`No Risk` / `Risk`) utilizando variables espectrales, espaciales y ambientales obtenidas a partir de imágenes satelitales y datos geográficos.

SVM resulta especialmente interesante para este caso porque:

- El dataset contiene principalmente variables numéricas continuas.
- Existen relaciones complejas y potencialmente no lineales entre vegetación, urbanización, humedad y riesgo térmico.
- Las clases objetivo se encuentran relativamente balanceadas.
- El número de variables es moderado y contiene información espacial y espectral relevante para la clasificación.

Además, SVM es un modelo sensible a la escala de las variables, ya que basa su funcionamiento en distancias matemáticas entre observaciones. Por este motivo, será necesario aplicar escalado (`StandardScaler`) antes del entrenamiento del modelo.

Finalmente, se evaluará el impacto de incluir o excluir las coordenadas geográficas (`latitude`, `longitude`) para analizar hasta qué punto el modelo depende de patrones espaciales explícitos frente a variables ambientales derivadas.

</div>

In [3]:
# ============================================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================================

# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# División de datos
from sklearn.model_selection import train_test_split

# Preprocesamiento
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Modelo
from sklearn.svm import SVC

# Métricas de evaluación
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay
)
from sklearn.model_selection import GridSearchCV

# Configuración visual
sns.set_style("whitegrid")
pd.set_option('display.float_format', '{:.4f}'.format)

In [1]:
# ============================================================
# CARGA DEL DATASET DE MODELADO
# ============================================================

# Cargamos el dataset final preparado durante el EDA
# y el proceso de feature engineering.

df_bcn = pd.read_csv('../data/processed/dataset_modeling.csv')
df_bcn.shape

(69683, 14)

In [6]:
# ============================================================
# SEPARACIÓN DE FEATURES Y TARGET
# ============================================================

# Separamos el dataset en:
# - X: variables predictoras que utilizará el modelo.
# - y: variable objetivo que queremos predecir.
#
# Esta separación debe hacerse antes del train/test split para mantener clara la diferencia entre información de entrada
# y variable a predecir.

X = df_bcn.drop(columns=['uhi_risk'])
y = df_bcn['uhi_risk']

# Verificamos dimensiones
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (69683, 13)
y shape: (69683,)


In [7]:
# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

# Dividimos el dataset en conjuntos de entrenamiento y prueba.
#
# - Train: utilizado para entrenar el modelo.
# - Test: utilizado para evaluar el rendimiento sobre datos no vistos.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Verificamos dimensiones
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (55746, 13)
X_test: (13937, 13)
y_train: (55746,)
y_test: (13937,)


In [8]:
# ============================================================
# DEFINICIÓN DEL PREPROCESAMIENTO
# ============================================================

# Definimos las variables numéricas y categóricas.
#
# - Las variables numéricas serán escaladas mediante StandardScaler.
# - La variable categórica season será transformada mediante One Hot Encoding.

numeric_features = [
    'latitude',
    'longitude',
    'elevation',
    'mndwi',
    'nbr',
    'ndbi',
    'ndmi',
    'ndvi',
    'ndwi',
    'nir',
    'swir1',
    'swir2'
]

categorical_features = [
    'season'
]

# Construimos el preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first'), categorical_features)
    ]
)

# Verificamos configuración
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

## Observación

Se aplicó `StandardScaler` sobre las variables numéricas debido a que SVM es un algoritmo sensible a la escala de los datos, ya que basa su funcionamiento en distancias matemáticas entre observaciones. El escalado permite que todas las variables contribuyan de forma equilibrada al entrenamiento del modelo.

Por otro lado, la variable categórica `season` fue transformada mediante `OneHotEncoder`, ya que las estaciones del año no representan una relación numérica lineal real entre categorías. Esta codificación evita introducir relaciones ordinales artificiales dentro del modelo.

El preprocesamiento se integró dentro de un `ColumnTransformer`, permitiendo aplicar automáticamente el tratamiento adecuado a cada tipo de variable de forma organizada y reproducible.

</div>

In [9]:
# ============================================================
# CREACIÓN DEL PIPELINE SVM
# ============================================================

# Creamos un pipeline que integra el preprocesamiento y el modelo SVM en un único flujo de entrenamiento.
#
# El preprocesador se ajustará solo con X_train cuando ejecutemos fit(), evitando data leakage.

svm_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', SVC(random_state=42))
    ]
)

svm_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [10]:
# ============================================================
# ENTRENAMIENTO DEL MODELO SVM
# ============================================================

# Entrenamos el pipeline completo utilizando únicamente
# los datos de entrenamiento.
#
# Durante este proceso:
# - StandardScaler aprende medias y desviaciones.
# - OneHotEncoder aprende las categorías.
# - SVM aprende los patrones del problema.

svm_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [11]:
# ============================================================
# PREDICCIONES DEL MODELO
# ============================================================

# Generamos predicciones sobre el conjunto de prueba
# para evaluar el rendimiento del modelo en datos no vistos.

y_pred = svm_pipeline.predict(X_test)

# Mostramos las primeras predicciones
y_pred[:10]

array([0, 1, 0, 0, 1, 0, 1, 0, 1, 1])

In [12]:
# ============================================================
# MÉTRICAS DE EVALUACIÓN DEL MODELO
# ============================================================

# Evaluamos el rendimiento del modelo utilizando
# métricas de clasificación.

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report")
print("Modelo con coordenadas geografica\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.7110568989022028

Classification Report
Modelo con coordenadas geografica

              precision    recall  f1-score   support

           0       0.71      0.66      0.69      6613
           1       0.71      0.75      0.73      7324

    accuracy                           0.71     13937
   macro avg       0.71      0.71      0.71     13937
weighted avg       0.71      0.71      0.71     13937



<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

## **Observación sobre las metricas**

En este proyecto se prioriza especialmente el recall de la clase `Risk`, ya que resulta más crítico detectar correctamente zonas con riesgo UHI que clasificar erróneamente algunas zonas seguras como áreas de riesgo.

Desde el punto de vista del fenómeno Urban Heat Island (UHI), un falso negativo implicaría no identificar una zona potencialmente afectada por acumulación anómala de calor nocturno, mientras que un falso positivo tendría un impacto menos severo durante el análisis exploratorio y preventivo.

Por este motivo, la evaluación del modelo no se centra únicamente en **accuracy**, sino también en métricas como **recall y F1-score** para la clase de riesgo.

</div>

--------------------------------------------------- **Optimizando modelo con coordenadas** ----------------------------------------------------------------

In [13]:
# ============================================================
#  OPTIMIZACIÓN SVM CON COORDENADAS
# ============================================================

# Buscamos la mejor combinación de hiperparámetros
# para maximizar el rendimiento del modelo SVM.

param_grid = {
    'classifier__C': [0.1, 1, 10],
    'classifier__kernel': ['linear', 'rbf'],
    'classifier__gamma': ['scale', 'auto']
}

grid_svm = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid_svm.fit(X_train, y_train)

Fitting 3 folds for each of 12 candidates, totalling 36 fits


[CV] END classifier__C=0.1, classifier__gamma=scale, classifier__kernel=linear; total time= 1.4min
[CV] END classifier__C=0.1, classifier__gamma=scale, classifier__kernel=linear; total time= 1.4min
[CV] END classifier__C=0.1, classifier__gamma=scale, classifier__kernel=linear; total time= 1.4min
[CV] END classifier__C=0.1, classifier__gamma=scale, classifier__kernel=rbf; total time= 1.9min
[CV] END classifier__C=0.1, classifier__gamma=scale, classifier__kernel=rbf; total time= 1.9min
[CV] END classifier__C=0.1, classifier__gamma=scale, classifier__kernel=rbf; total time= 1.9min
[CV] END classifier__C=0.1, classifier__gamma=auto, classifier__kernel=linear; total time= 1.4min
[CV] END classifier__C=0.1, classifier__gamma=auto, classifier__kernel=linear; total time= 1.4min
[CV] END classifier__C=0.1, classifier__gamma=auto, classifier__kernel=linear; total time= 1.4min
[CV] END classifier__C=0.1, classifier__gamma=auto, classifier__kernel=rbf; total time= 1.9min
[CV] END classifier__C=0.1

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__C': [0.1, 1, ...], 'classifier__gamma': ['scale', 'auto'], 'classifier__kernel': ['linear', 'rbf']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate

In [14]:
# ============================================================
# EVALUACIÓN DEL MODELO SVM OPTIMIZADO CON COORDENADAS
# ============================================================

# Evaluamos el rendimiento del mejor modelo SVM
# optimizado utilizando latitude y longitude.

best_svm_coords = grid_svm.best_estimator_

y_pred_best_coords = best_svm_coords.predict(X_test)

print("Accuracy modelo SVM optimizado con coordenadas:", accuracy_score(y_test, y_pred_best_coords))

print("\nClassification Report - SVM optimizado con coordenadas:\n")

print(classification_report(y_test, y_pred_best_coords))

Accuracy modelo SVM optimizado con coordenadas: 0.7172275238573581

Classification Report - SVM optimizado con coordenadas:

              precision    recall  f1-score   support

           0       0.72      0.67      0.69      6613
           1       0.72      0.76      0.74      7324

    accuracy                           0.72     13937
   macro avg       0.72      0.71      0.72     13937
weighted avg       0.72      0.72      0.72     13937



<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

## **Análisis coordenadas geográfica**

Dado que las variables `latitude` y `longitude` podrían introducir una fuerte dependencia espacial dentro del modelo, se decidió realizar entrenamientos adicionales eliminando las coordenadas geográficas.

El objetivo de esta comparación es evaluar si el modelo es capaz de aprender el fenómeno Urban Heat Island (UHI) utilizando principalmente variables térmicas, espectrales y ambientales, y no únicamente la localización espacial de los registros.

Este análisis permitirá determinar hasta qué punto las coordenadas aportan información relevante o si podrían estar generando un sobreajuste espacial del modelo.

La que nos gustaria saber es: **“¿El modelo aprende el fenómeno climático o simplemente memoriza ubicación?”**

**NOTA:** Aunque `elevation` presenta relación con la temperatura superficial, se decidió mantenerla en el modelo debido a que representa una
característica física real del entorno urbano y no únicamente una referencia espacial directa.
</div>

In [15]:
# ============================================================
# ELIMINACIÓN DE COORDENADAS GEOGRÁFICAS
# ============================================================

# Creamos una nueva versión del dataset eliminando
# latitude y longitude para evaluar el impacto de
# la dependencia espacial en el modelo.

X_no_geo = X.drop(columns=['latitude', 'longitude'])

# Verificamos las columnas restantes
X_no_geo.columns.tolist()

['elevation',
 'mndwi',
 'nbr',
 'ndbi',
 'ndmi',
 'ndvi',
 'ndwi',
 'nir',
 'swir1',
 'swir2',
 'season']

In [16]:
# ============================================================
# TRAIN / TEST SPLIT SIN COORDENADAS
# ============================================================

# Dividimos nuevamente el dataset sin coordenadas geográficas.

X_train_ng, X_test_ng, y_train_ng, y_test_ng = train_test_split(
    X_no_geo,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Verificamos dimensiones
print("X_train:", X_train_ng.shape)
print("X_test:", X_test_ng.shape)
print("y_train:", y_train_ng.shape)
print("y_test:", y_test_ng.shape)

X_train: (55746, 11)
X_test: (13937, 11)
y_train: (55746,)
y_test: (13937,)


In [17]:
# ============================================================
# PREPROCESAMIENTO SIN COORDENADAS
# ============================================================

numeric_features_ng = [
    'elevation',
    'mndwi',
    'nbr',
    'ndbi',
    'ndmi',
    'ndvi',
    'ndwi',
    'nir',
    'swir1',
    'swir2'
]

categorical_features_ng = [
    'season'
]

preprocessor_ng = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features_ng),
        ('cat', OneHotEncoder(drop='first'), categorical_features_ng)
    ]
)

svm_pipeline_ng = Pipeline(
    steps=[
        ('preprocessor', preprocessor_ng),
        ('classifier', SVC(random_state=42))
    ]
)

svm_pipeline_ng.fit(X_train_ng, y_train_ng)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [18]:
# ============================================================
# PREDICCIONES DEL MODELO SVM SIN COORDENADAS
# ============================================================

# Generamos predicciones utilizando el modelo entrenado sin latitude ni longitude.

y_pred_ng = svm_pipeline_ng.predict(X_test_ng)

# Mostramos las primeras predicciones
y_pred_ng[:10]

array([0, 1, 0, 0, 1, 0, 1, 1, 0, 1])

In [19]:
# ============================================================
# MÉTRICAS DE EVALUACIÓN DEL MODELO SIN COORDENADAS
# ============================================================

# Evaluamos el rendimiento del modelo utilizando
# métricas de clasificación.

print("Accuracy:", accuracy_score(y_test_ng, y_pred_ng))

print("\nClassification Report")
print("Modelo SIN coordenadas geografica\n")

print(classification_report(y_test_ng, y_pred_ng))

Accuracy: 0.6593241013130516

Classification Report
Modelo SIN coordenadas geografica

              precision    recall  f1-score   support

           0       0.66      0.59      0.62      6613
           1       0.66      0.72      0.69      7324

    accuracy                           0.66     13937
   macro avg       0.66      0.66      0.66     13937
weighted avg       0.66      0.66      0.66     13937



<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

## **Observación comparación modelos SVM**

Se entrenaron dos versiones del modelo SVM:

- Un modelo utilizando coordenadas geográficas (`latitude` y `longitude`).
- Un modelo eliminando las coordenadas para evaluar el impacto de la dependencia espacial en el aprendizaje del modelo.

| Modelo | Accuracy | Recall (Risk) | F1-score (Risk) |
|---|---|---|---|
| Con coordenadas | 0.71 | 0.75 | 0.73 |
| Sin coordenadas | 0.66 | 0.72 | 0.69 |

La eliminación de las coordenadas produce una disminución moderada del rendimiento, aunque el modelo no se derrumba. Esto resulta relevante, ya que sugiere que el modelo no está simplemente memorizando ubicaciones geográficas específicas.

Si la caída hubiese sido extrema (por ejemplo, de 0.71 a 0.52), podría interpretarse como una fuerte dependencia espacial o sobreajuste a la localización. Sin embargo, la reducción observada es relativamente contenida, lo que indica que las coordenadas ayudan al modelo, pero no son la única fuente de información relevante.

Incluso sin `latitude` y `longitude`, el modelo sigue siendo capaz de detectar patrones asociados al fenómeno Urban Heat Island mediante variables térmicas y espectrales como temperatura superficial, vegetación, humedad y urbanización.

Además, el recall de la clase `Risk` continúa siendo razonablemente alto (`0.72`), lo que indica que el modelo mantiene capacidad para identificar zonas de riesgo térmico aun sin información espacial explícita.

En conjunto, esta comparación aporta mayor robustez al análisis, ya que demuestra que el modelo mejora al incorporar información geográfica, pero continúa aprendiendo patrones ambientales reales incluso cuando las coordenadas son eliminadas.

</div>

<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

## **Decisión final del modelo a optimizar**

Aunque el modelo SVM que incorpora coordenadas geográficas (`latitude` y `longitude`) obtiene mejores métricas globales, se decidió continuar el proceso de optimización utilizando la versión **sin coordenadas geográficas**.

Esta decisión busca reducir la dependencia espacial del modelo y favorecer un aprendizaje basado principalmente en variables ambientales y térmicas relacionadas con el fenómeno Urban Heat Island (UHI).

El objetivo no es que el modelo aprenda únicamente a reconocer zonas específicas del mapa, sino que sea capaz de identificar patrones físicos asociados al riesgo térmico urbano, tales como:

- Temperatura superficial.
- Cobertura vegetal.
- Humedad.
- Urbanización.
- Diferencias térmicas.

La comparación entre ambos modelos mostró que, al eliminar las coordenadas, el rendimiento disminuye de forma moderada pero el modelo no se derrumba. Esto indica que las variables ambientales continúan aportando información predictiva relevante incluso sin información espacial explícita.

En consecuencia, se considera que **el modelo sin coordenadas ofrece una aproximación más robusta e interpretable desde el punto de vista ambiental**, manteniendo además métricas razonables para la detección de zonas de riesgo UHI.

</div>